In [ ]:
# IMPORTANTE: Si estás ejecutando este cuadernillo en Google Colab, 
# descomenta y ejecuta estas líneas para montar tu Google Drive.
# Asegúrate de subir tus archivos CSV a una carpeta dentro de tu Drive
# y luego cambia la ruta en los pd.read_csv() más abajo apuntando a tu Drive.

from google.colab import drive
drive.mount('/content/drive')

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

# Laboratorio 5: Modelos en PyTorch
## Parte 1: Regresión Lineal Multivariable
En este cuadernillo unificaremos los conceptos de Regresión Lineal, Regresión Logística (Binaria) y Clasificación Multiclase utilizando Redes Neuronales con PyTorch, aplicando `Dataset`, `DataLoader` y validación.

In [ ]:
import torch
import pandas as pd
import numpy as np

# 1. Dispositivo disponible (tal como lo usa el docente)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

### 1.1 Preprocesamiento con Pandas
Se extrae la variable *target* 'shares', se descartan los textos que no tienen valor numérico y se divide el dataset al 80/20.

In [ ]:
# Leer el archivo csv en un dataframe
df_news = pd.read_csv('/content/drive/MyDrive/LABSIS420/OnlineNewsPopularity.csv')

# Limpiamos nombres de columnas quitando espacios iniciales o finales
df_news.columns = df_news.columns.str.strip()

# Eliminamos atributos url y timedelta, no contribuyen
df_news = df_news.drop(['url', 'timedelta'], axis=1)

X_pandas = df_news.drop(['shares'], axis=1).values
Y_pandas = df_news['shares'].values

m = len(Y_pandas)
print(f"Total de ejemplos (m): {m}")
print(f"Total de propiedades (n): {X_pandas.shape[1]}")

# Split: 80% entrenamiento (train) y 20% prueba (test)
limite = int(0.8 * m)
X_train_raw = X_pandas[:limite]
X_test_raw = X_pandas[limite:]
Y_train_raw = Y_pandas[:limite]
Y_test_raw = Y_pandas[limite:]

# Normalización: crucial para el costo
X_mean = X_train_raw.mean(axis=0)
X_std = X_train_raw.std(axis=0) + 1e-8

X_train = (X_train_raw - X_mean) / X_std
X_test = (X_test_raw - X_mean) / X_std

# Para evitar que el gradiente explote ('nan'), también normalizamos Y
y_mean = Y_train_raw.mean()
y_std = Y_train_raw.std() + 1e-8

y_train_norm = (Y_train_raw - y_mean) / y_std
y_test_norm = (Y_test_raw - y_mean) / y_std

# En regresión, Y debe ser un vector columna
y_train = y_train_norm.astype(np.float32).reshape(-1, 1)
y_test = y_test_norm.astype(np.float32).reshape(-1, 1)
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

### 1.2 Dataset y DataLoader
Misma estructura que la clase del cuaderno `03 pytorch_datasets.ipynb`.

In [ ]:
class DatasetRegresion(torch.utils.data.Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X).float().to(device)
        self.Y = torch.from_numpy(Y).float().to(device)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, ix):
        return self.X[ix], self.Y[ix]

# Instanciamos
dataset_train_reg = DatasetRegresion(X_train, y_train)
dataset_test_reg = DatasetRegresion(X_test, y_test)

batch_size = 2000
dataloader_train_reg = torch.utils.data.DataLoader(dataset_train_reg, batch_size=batch_size, shuffle=True)
dataloader_test_reg = torch.utils.data.DataLoader(dataset_test_reg, batch_size=batch_size, shuffle=False)
print("DataLoaders creados correctamente.")

In [ ]:
import matplotlib.pyplot as plt

# --- 1.3 Diseño del Modelo (Deep Learning - Multicapa) ---
# n es el número de características. Lo sacamos del shape de X_train
D_in = X_train.shape[1] 
H1 = 100 # Primera capa oculta
H2 = 50  # Segunda capa oculta
D_out = 1 # Salida: 'shares'

# Utilizamos nn.Sequential con capas ocultas y funciones de activación ReLU
model_reg = torch.nn.Sequential(
    torch.nn.Linear(D_in, H1),
    torch.nn.ReLU(),
    torch.nn.Linear(H1, H2),
    torch.nn.ReLU(),
    torch.nn.Linear(H2, D_out)
).to(device)

# --- 1.4 Optimizador y Función de Pérdida ---
# Para regresión (números continuos) la regla es usar el Error Cuadrático Medio (MSELoss)
criterion = torch.nn.MSELoss() 
# Optimizador SGD con un learning rate más pequeño por seguridad
optimizer = torch.optim.SGD(model_reg.parameters(), lr=0.005)

# --- 1.5 Bucle de Entrenamiento (Training Loop) ---
epochs = 100
log_each = 10
l = [] # Aquí guardaremos el histórico de "loss" o costo
model_reg.train() # Ponemos el modelo en modo entrenamiento

print("Iniciando Entrenamiento...")
for e in range(1, epochs + 1):
    _l = []
    # Iteramos por batches en el dataloader como en "03 pytorch_datasets"
    for x_b, y_b in dataloader_train_reg:
        # forward (Hacia adelante, calcular predicción)
        y_pred = model_reg(x_b)
        
        # loss (Pérdida/Costo)
        loss = criterion(y_pred, y_b)
        _l.append(loss.item())
        
        # ponemos a cero los gradientes
        optimizer.zero_grad()
        
        # Backprop (Cálculo de derivadas automáticamente)
        loss.backward()
        
        # update de los pesos (Ajustar thetas)
        optimizer.step()

    # Guardar promedio de costo en la época
    l.append(np.mean(_l))
    
    if not e % log_each:
        print(f"Epoch {e}/{epochs} - Costo (Loss): {np.mean(l):.5f}")

print("Entrenamiento finalizado.")

# --- 1.6 Guardado del Modelo (Checkpoint) ---
# Tal como se vió en "04 pytorch_save.ipynb"
PATH = './checkpoint_regresion.pt'
torch.save(model_reg.state_dict(), PATH)
print(f"Modelo guardado en: {PATH}")

# --- 1.7 Gráfico de Costo ---
plt.plot(range(1, epochs + 1), l)
plt.title('Gráfico de Costo - Regresión Lineal Multivariable')
plt.xlabel('Iteraciones (Épocas)')
plt.ylabel('Costo (Loss)')
plt.grid(True)
plt.show()

In [ ]:
# --- 1.8 Evaluación y Validaciones (Prueba de efectividad) ---
# Cambiamos a modo evaluación (apaga ciertos comportamientos como Dropout si los hubiera)
model_reg.eval()

# Según las instrucciones: "realizar por lo menos 100 predicciones que demuestren la efectividad"
# Tomaremos 100 ejemplos de nuestro conjunto de test (el 20% que no vió en el entrenamiento)
X_prueba = torch.from_numpy(X_test[:100]).float().to(device)
y_prueba_real = y_test[:100]

with torch.no_grad(): # Fundamental: apagamos el cálculo de gradientes para no gastar memoria
    y_prediccion = model_reg(X_prueba)

# Pasamos los resultados a la CPU y DESNORMALIZAMOS para ver los valores a escala natural
y_pred_numpy = y_prediccion.cpu().numpy()
y_pred_desnormalizada = (y_pred_numpy * y_std) + y_mean

# y_prueba_real ya es un numpy array desde arriba, solo lo desnormalizamos
y_real_desnormalizada = (y_prueba_real * y_std) + y_mean

print(f"--- Realizadas {len(y_pred_numpy)} predicciones exitosamente ---")
print("Mostrando los primeros 15 resultados (Real vs Predicho):")
for i in range(15):
    print(f"Ejemplo {i+1} | Valor Real (Shares): {y_real_desnormalizada[i][0]:.0f} | Predicción: {y_pred_desnormalizada[i][0]:.0f}")

# También calculamos el Error Cuadrático Medio Global en TODOS los datos de Test
errores_test = []
for x_b, y_b in dataloader_test_reg:
    with torch.no_grad():
        y_p = model_reg(x_b)
        costo_test = criterion(y_p, y_b)
        errores_test.append(costo_test.item())

print(f"\nCosto Promedio en Datos de Prueba (Test Loss): {np.mean(errores_test):.5f}")

---
## Parte 2: Regresión Logística Binaria (Clasificación)
En esta segunda parte, buscaremos clasificar un dataset utilizando Regresión Logística mediante Redes Neuronales. Usaremos el dataset de "Buzz in Social Media" para determinar mediante las propiedades si un tema será popular o no (Clase 1 o Clase 0).
</VSCode.Cell>
<VSCode.Cell language="python">
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

# Asumiremos la ruta local. Cambiar si estás en Colab con Drive.
archivo_buzz = 'Twitter.data'

# Este dataset no trae nombre de columnas adentro, Pandas le pondría los valores de la primera fila.
# Lo leemos sin encabezados (header=None) para no perder el primer dato.
df_buzz = pd.read_csv(archivo_buzz, header=None)

# 2.1 Preprocesamiento: Transformar una tarea continua a Clasificación Binaria
# Según el archivo Twitter.names, la última columna (77) es la "Anotación" (Target a predecir).
X_buzz_pandas = df_buzz.iloc[:, :-1].values # Tomamos desde la columna 0 hasta la 76
Y_buzz_pandas_raw = df_buzz.iloc[:, -1].values  # Tomamos la última columna (77)

# Como este número es continuo (promedio de discusiones), lo convertiremos en Clases (0 y 1).
# Si está por encima del promedio global, es Popular (1). Si no, es No Popular (0).
umbral = Y_buzz_pandas_raw.mean()
Y_buzz_clases = (Y_buzz_pandas_raw > umbral).astype(int)

m = len(Y_buzz_clases)
print(f"Total de ejemplos (m): {m} (El Ing. pedía >= 20000)")
print(f"Total de propiedades (n): {X_buzz_pandas.shape[1]} (El Ing. pedía >= 5)")

# Contamos cuántos de cada clase hay
unicos, conteos = np.unique(Y_buzz_clases, return_counts=True)
print(f"Clase 0 (No Popular): {conteos[0]}")
print(f"Clase 1 (Popular): {conteos[1]}")

In [ ]:
# 2.2 Split: Dividir 80% Entrenamiento y 20% Prueba
limite_buzz = int(0.8 * m)

X_train_buzz_raw = X_buzz_pandas[:limite_buzz]
X_test_buzz_raw = X_buzz_pandas[limite_buzz:]
Y_train_buzz = Y_buzz_clases[:limite_buzz]
Y_test_buzz = Y_buzz_clases[limite_buzz:]

# Normalización obligatoria para redes profundas
X_mean_buzz = X_train_buzz_raw.mean(axis=0)
X_std_buzz = X_train_buzz_raw.std(axis=0) + 1e-8

X_train_buzz = (X_train_buzz_raw - X_mean_buzz) / X_std_buzz
X_test_buzz = (X_test_buzz_raw - X_mean_buzz) / X_std_buzz

# A diferencia de Regresión, para Clasificación Binaria (BCE Loss) Y debe ser Float [0.0 o 1.0] 
# Y seguir en vector columna
y_train_buzz = Y_train_buzz.astype(np.float32).reshape(-1, 1)
y_test_buzz = Y_test_buzz.astype(np.float32).reshape(-1, 1)
X_train_buzz = X_train_buzz.astype(np.float32)
X_test_buzz = X_test_buzz.astype(np.float32)

# 2.3 Reutilizamos la misma clase Dataset
dataset_train_bin = DatasetRegresion(X_train_buzz, y_train_buzz)
dataset_test_bin = DatasetRegresion(X_test_buzz, y_test_buzz)

# DataLoader (tamaño 2000 para que aprenda súper rápido y no salten picos feos)
batch_size_bin = 2000
dataloader_train_bin = torch.utils.data.DataLoader(dataset_train_bin, batch_size=batch_size_bin, shuffle=True)
dataloader_test_bin = torch.utils.data.DataLoader(dataset_test_bin, batch_size=batch_size_bin, shuffle=False)
print("DataLoaders Binarios listos.")

In [ ]:
# --- 2.4 Diseño de la Red Neuronal (Clasificación Binaria) ---
D_in_bin = X_train_buzz.shape[1] # 77 atributos
H_bin = 150 # Una buena cantidad de neuronas para buscar patrones base
D_out_bin = 1 # Respuesta Binaria (Cero o Uno)

# El ingeniero pidió Regresión Logística
# Matemáticamente es: entrada -> (Capas ocultas opcionales) -> Función Sigmoide.
model_bin = torch.nn.Sequential(
    torch.nn.Linear(D_in_bin, H_bin),
    torch.nn.ReLU(),
    torch.nn.Linear(H_bin, D_out_bin),
    torch.nn.Sigmoid() # ¡ESTA es la magia de la regresión logística binaria! Devuelve Probabilidad de [0 a 1]
).to(device)

# --- 2.5 Optimizador y Función de Pérdida ---
# Para ceros y unos usamos BCELoss (Binary Cross Entropy)
criterion_bin = torch.nn.BCELoss() 
# Optimizador Adam para que aprenda estable y no produzca picos ruidosos
optimizer_bin = torch.optim.Adam(model_bin.parameters(), lr=0.005)

# --- 2.6 Bucle de Entrenamiento ---
epochs_bin = 250 # Con Adam y un buen batch_size, 250 serán super rápidas y suficientes.
log_each_bin = 50
l_bin = []
model_bin.train()

print("Iniciando Entrenamiento para Regresión Logística (Binaria)...")
for e in range(1, epochs_bin + 1):
    _l = []
    for x_b, y_b in dataloader_train_bin:
        y_pred = model_bin(x_b)
        loss = criterion_bin(y_pred, y_b) # Ojo: Aquí ya evalúa probabilidades vs (0 o 1) reales
        
        _l.append(loss.item())
        optimizer_bin.zero_grad()
        loss.backward()
        optimizer_bin.step()

    l_bin.append(np.mean(_l))
    if not e % log_each_bin:
        print(f"Epoch {e}/{epochs_bin} - Costo Binario(Loss): {np.mean(l_bin):.5f}")

print("Entrenamiento Binario finalizado.")

# --- Guardar modelo ---
PATH_BIN = './checkpoint_clasificacion_binaria.pt'
torch.save(model_bin.state_dict(), PATH_BIN)
print(f"Modelo Binario guardado en: {PATH_BIN}")

# --- Gráfico de Costo ---
plt.plot(range(1, epochs_bin + 1), l_bin, color='green')
plt.title('Gráfico de Costo - Regresión Logística Binaria (Clasificación)')
plt.xlabel('Iteraciones (Épocas)')
plt.ylabel('Costo (BCELoss)')
plt.grid(True)
plt.show()

In [ ]:
# --- 2.7 Validación y Cálculo de Precisión (Accuracy) ---
model_bin.eval() # Modo evaluación

correctos = 0
total_ejemplos = len(dataset_test_bin)

with torch.no_grad():
    # Iteramos sobre todos los datos de validación
    for x_b, y_b in dataloader_test_bin:
        # Pide probabilidades [0.1, 0.99, 0.05, 0.8]
        probabilidades = model_bin(x_b)
        
        # Redondea las probabilidades para forzarlas a Clase 0 o Clase 1
        # Ej: 0.99 -> 1.0 (Popular), 0.05 -> 0.0 (No popular)
        prediccion_clase = (probabilidades >= 0.5).float()
        
        # Cuenta cuántas coinciden con los valores reales (y_b)
        correctos += (prediccion_clase == y_b).sum().item()

# Cálculo del Porcentaje
accuracy = (correctos / total_ejemplos) * 100
print(f"La precisión (Accuracy) del modelo en los datos de Prueba es: {accuracy:.2f}%")

# Visualización rápida de los primeros 15 datos
X_prueba_bin = torch.from_numpy(X_test_buzz[:15]).float().to(device)
y_real_bin = y_test_buzz[:15]

with torch.no_grad():
     clasificaciones = (model_bin(X_prueba_bin) >= 0.5).float().cpu().numpy()

print("\n--- Analizando 15 Predicciones Individuales ---")
for i in range(15):
    txt_real = "Popular (1)" if y_real_bin[i][0] == 1 else "No popular (0)"
    txt_pred = "Popular (1)" if clasificaciones[i][0] == 1 else "No popular (0)"
    flag = "✅" if y_real_bin[i][0] == clasificaciones[i][0] else "❌"
    print(f"Ejemplo {i+1} | Real: {txt_real} | Predicción: {txt_pred} {flag}")